# Capítulo 9: Tratamento de Dados e Visualização

**Disciplina:** FSC1189 - Algoritmo e Programação

**Tema:** Análise estatística, detecção de anomalias, interpolação e série temporal

Este notebook cobre técnicas essenciais para processamento de dados experimentais em Física.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.interpolate import interp1d, CubicSpline
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)  # Reprodutibilidade

## Bloco A: Geração de Dados Sintéticos Realistas

Simulamos dados de um experimento de pêndulo simples com ruído e anomalias.

In [ ]:
class ExperimentoPendulo:
    """Simula um experimento de medição do período do pêndulo simples."""
    
    def __init__(self, seed=42):
        np.random.seed(seed)
        self.g = 9.81  # m/s²
    
    def periodo_teorico(self, comprimento):
        """T = 2π√(L/g)"""
        return 2 * np.pi * np.sqrt(np.array(comprimento) / self.g)
    
    def gerar_medidas(self, comprimentos, num_medidas=15, sigma_relativo=0.01):
        """Gera medidas de período com ruído gaussiano e alguns outliers."""
        periodos_teoricos = self.periodo_teorico(comprimentos)
        
        dados = []
        for L, T_teorico in zip(comprimentos, periodos_teoricos):
            # Ruído gaussiano
            sigma = sigma_relativo * T_teorico
            medidas = np.random.normal(T_teorico, sigma, num_medidas)
            
            # Adicionar 1-2 outliers por comprimento
            num_outliers = np.random.randint(0, 2)
            if num_outliers > 0:
                idx_outliers = np.random.choice(num_medidas, num_outliers, replace=False)
                for idx in idx_outliers:
                    # Outlier é 5-10% maior/menor que valor teórico
                    medidas[idx] = T_teorico * (1 + np.random.uniform(0.05, 0.15) * np.sign(np.random.randn()))
            
            # Criar linhas do DataFrame
            for i, medida in enumerate(medidas):
                dados.append({
                    'Comprimento (m)': L,
                    'Período (s)': medida,
                    'Medida': i + 1
                })
        
        return pd.DataFrame(dados)

# Gerar dados
experimento = ExperimentoPendulo()
comprimentos = np.array([0.3, 0.5, 0.8, 1.0, 1.2])
df_pendulo = experimento.gerar_medidas(comprimentos, num_medidas=15)

print("=== DADOS DO EXPERIMENTO PÊNDULO ===")
print(f"Número de comprimentos testados: {len(comprimentos)}")
print(f"Número de medidas por comprimento: 15")
print(f"Total de registros: {len(df_pendulo)}")
print()
print("Primeiras linhas dos dados:")
print(df_pendulo.head(10))

## Bloco B: Estatística Descritiva

Análise estatística básica dos dados de período.

In [ ]:
# Estatísticas por comprimento
stats_by_L = df_pendulo.groupby('Comprimento (m)')['Período (s)'].agg([
    ('Média', 'mean'),
    ('Mediana', 'median'),
    ('Desvio Padrão', 'std'),
    ('Min', 'min'),
    ('Max', 'max'),
    ('Amplitude', lambda x: x.max() - x.min())
])

print("\n=== ESTATÍSTICAS DESCRITIVAS ===")
print(stats_by_L.to_string())

# Visualização
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histograma dos períodos
ax = axes[0]
ax.hist(df_pendulo['Período (s)'], bins=20, color='skyblue', edgecolor='black', alpha=0.7)
mu, sigma = df_pendulo['Período (s)'].mean(), df_pendulo['Período (s)'].std()

# Ajustar distribuição normal
x_norm = np.linspace(df_pendulo['Período (s)'].min(), df_pendulo['Período (s)'].max(), 100)
y_norm = stats.norm.pdf(x_norm, mu, sigma) * len(df_pendulo) * (stats_by_L.index[1] - stats_by_L.index[0]) / 2
ax.plot(x_norm, y_norm, 'r-', linewidth=2, label=f'Normal (μ={mu:.3f}, σ={sigma:.4f})')
ax.set_xlabel('Período (s)', fontsize=11)
ax.set_ylabel('Frequência', fontsize=11)
ax.set_title('Distribuição de Períodos', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Box plot por comprimento
ax = axes[1]
df_pendulo.boxplot(column='Período (s)', by='Comprimento (m)', ax=ax)
ax.set_xlabel('Comprimento (m)', fontsize=11)
ax.set_ylabel('Período (s)', fontsize=11)
ax.set_title('Box Plot: Período vs Comprimento', fontsize=12, fontweight='bold')
ax.get_figure().suptitle('')  # Remove título automático
plt.setp(ax.xaxis.get_majorticklabels(), rotation=0)

# Violin plot
ax = axes[2]
L_unique = sorted(df_pendulo['Comprimento (m)'].unique())
periodos_por_L = [df_pendulo[df_pendulo['Comprimento (m)'] == L]['Período (s)'].values for L in L_unique]
parts = ax.violinplot(periodos_por_L, positions=range(len(L_unique)), showmeans=True, showmedians=True)
ax.set_xticks(range(len(L_unique)))
ax.set_xticklabels([f'{L:.1f}' for L in L_unique])
ax.set_xlabel('Comprimento (m)', fontsize=11)
ax.set_ylabel('Período (s)', fontsize=11)
ax.set_title('Violin Plot: Distribuição por Comprimento', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Bloco C: Detecção de Outliers

Identificar e visualizar valores anômalos usando dois métodos: Z-score e IQR.

In [ ]:
def detectar_outliers_zscore(dados, threshold=2.5):
    """Detecta outliers usando Z-score."""
    z_scores = np.abs(stats.zscore(dados))
    return z_scores > threshold

def detectar_outliers_iqr(dados, fator=1.5):
    """Detecta outliers usando método IQR."""
    Q1 = np.percentile(dados, 25)
    Q3 = np.percentile(dados, 75)
    IQR = Q3 - Q1
    lower = Q1 - fator * IQR
    upper = Q3 + fator * IQR
    return (dados < lower) | (dados > upper)

# Detectar outliers
periodos = df_pendulo['Período (s)'].values
outliers_zscore = detectar_outliers_zscore(periodos, threshold=2.5)
outliers_iqr = detectar_outliers_iqr(periodos, fator=1.5)

# Criar máscara combinada
outliers_combinado = outliers_zscore | outliers_iqr

df_pendulo['Outlier_ZScore'] = outliers_zscore
df_pendulo['Outlier_IQR'] = outliers_iqr
df_pendulo['Outlier'] = outliers_combinado

print("=== DETECÇÃO DE OUTLIERS ===")
print(f"Outliers detectados por Z-score: {outliers_zscore.sum()}")
print(f"Outliers detectados por IQR: {outliers_iqr.sum()}")
print(f"Outliers totais (combinado): {outliers_combinado.sum()}")
print()
print("Índices dos outliers:")
print(df_pendulo[outliers_combinado][['Comprimento (m)', 'Período (s)', 'Outlier_ZScore', 'Outlier_IQR']])

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot mostrando outliers
ax = axes[0]
for L in sorted(df_pendulo['Comprimento (m)'].unique()):
    mask_L = df_pendulo['Comprimento (m)'] == L
    mask_normal = mask_L & ~outliers_combinado
    mask_outlier = mask_L & outliers_combinado
    
    ax.scatter(df_pendulo[mask_normal]['Comprimento (m)'],
              df_pendulo[mask_normal]['Período (s)'],
              s=60, alpha=0.6, color='blue', label='Normal' if L == sorted(df_pendulo['Comprimento (m)'].unique())[0] else '')
    ax.scatter(df_pendulo[mask_outlier]['Comprimento (m)'],
              df_pendulo[mask_outlier]['Período (s)'],
              s=100, marker='X', color='red', edgecolors='darkred', linewidth=2,
              label='Outlier' if L == sorted(df_pendulo['Comprimento (m)'].unique())[0] else '')

ax.set_xlabel('Comprimento (m)', fontsize=11)
ax.set_ylabel('Período (s)', fontsize=11)
ax.set_title('Detecção de Outliers (Z-score + IQR)', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Comparação antes/depois da limpeza
ax = axes[1]
T_com_outliers = df_pendulo['Período (s)'].values
T_sem_outliers = df_pendulo[~outliers_combinado]['Período (s)'].values

ax.hist(T_com_outliers, bins=15, alpha=0.6, label='Com outliers', color='red', edgecolor='black')
ax.hist(T_sem_outliers, bins=15, alpha=0.6, label='Sem outliers', color='green', edgecolor='black')
ax.axvline(T_com_outliers.mean(), color='red', linestyle='--', linewidth=2, label=f'Média com outliers: {T_com_outliers.mean():.4f}')
ax.axvline(T_sem_outliers.mean(), color='green', linestyle='--', linewidth=2, label=f'Média sem outliers: {T_sem_outliers.mean():.4f}')
ax.set_xlabel('Período (s)', fontsize=11)
ax.set_ylabel('Frequência', fontsize=11)
ax.set_title('Impacto dos Outliers nas Estatísticas', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Bloco D: Interpolação de Dados

Usar dados de refração de luz para demonstrar diferentes métodos de interpolação.

In [ ]:
# Dados de refração do vidro borosilicato (índice de refração vs comprimento de onda)
# Dados reais simplificados
lambda_dados = np.array([0.4, 0.45, 0.5, 0.55, 0.6, 0.7, 0.8])  # micrometros
n_dados = np.array([1.5320, 1.5241, 1.5175, 1.5119, 1.5074, 1.5013, 1.4968])  # índice de refração

print("=== DADOS DE REFRAÇÃO ===")
df_refracao = pd.DataFrame({'λ (μm)': lambda_dados, 'n': n_dados})
print(df_refracao.to_string(index=False))
print()

# Métodos de interpolação
lambda_interp = np.linspace(0.4, 0.8, 100)

# 1. Interpolação linear
f_linear = interp1d(lambda_dados, n_dados, kind='linear')
n_linear = f_linear(lambda_interp)

# 2. Interpolação cúbica (SciPy)
f_cubic = interp1d(lambda_dados, n_dados, kind='cubic')
n_cubic = f_cubic(lambda_interp)

# 3. Spline Cúbica
cs = CubicSpline(lambda_dados, n_dados)
n_spline = cs(lambda_interp)

# Calcular diferenças
diferenca_linear_cubic = np.abs(n_linear - n_cubic)
diferenca_linear_spline = np.abs(n_linear - n_spline)

print(f"Diferença máxima (Linear vs Cúbica): {np.max(diferenca_linear_cubic):.2e}")
print(f"Diferença máxima (Linear vs Spline): {np.max(diferenca_linear_spline):.2e}")

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Painel 1: Comparação de interpolações
ax = axes[0]
ax.plot(lambda_interp, n_linear, 'b-', linewidth=2, label='Linear', alpha=0.7)
ax.plot(lambda_interp, n_cubic, 'g-', linewidth=2, label='Cúbica', alpha=0.7)
ax.plot(lambda_interp, n_spline, 'r--', linewidth=2, label='Spline Cúbica', alpha=0.7)
ax.plot(lambda_dados, n_dados, 'ko', markersize=8, label='Dados experimentais', zorder=5)
ax.set_xlabel('Comprimento de onda λ (μm)', fontsize=11)
ax.set_ylabel('Índice de refração n', fontsize=11)
ax.set_title('Métodos de Interpolação: Índice de Refração', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Painel 2: Diferenças entre métodos
ax = axes[1]
ax.semilogy(lambda_interp, diferenca_linear_cubic, 'b-', linewidth=2.5, label='|Linear - Cúbica|')
ax.semilogy(lambda_interp, diferenca_linear_spline, 'g-', linewidth=2.5, label='|Linear - Spline|')
ax.set_xlabel('Comprimento de onda λ (μm)', fontsize=11)
ax.set_ylabel('Diferença (escala log)', fontsize=11)
ax.set_title('Erro entre Métodos de Interpolação', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

## Bloco E: Análise de Série Temporal

Simulação de dados de temperatura com suavização e remoção de drift.

In [ ]:
# Gerar série temporal de temperatura
num_pontos = 200
t_array = np.linspace(0, 24, num_pontos)  # 24 horas

# Componentes da temperatura
T_base = 20  # temperatura média (°C)
T_ciclo = 8 * np.sin(2 * np.pi * t_array / 24)  # variação diária (amplitude 8°C)
T_drift = 0.05 * t_array  # drift linear do sensor (+0.05°C/hora)
T_ruido = np.random.normal(0, 0.5, num_pontos)  # ruído gaussiano
T_spike = np.zeros_like(t_array)

# Adicionar alguns spikes (anomalias)
spike_indices = np.random.choice(num_pontos, 3, replace=False)
for idx in spike_indices:
    T_spike[idx] = np.random.uniform(3, 5)  # spikes de 3-5°C

T_medido = T_base + T_ciclo + T_drift + T_ruido + T_spike

# Criar DataFrame
df_temp = pd.DataFrame({
    'Tempo (h)': t_array,
    'Temperatura (°C)': T_medido,
    'Temp_Ideal': T_base + T_ciclo
})

print("=== ANÁLISE DE SÉRIE TEMPORAL ===")
print(f"Número de pontos: {len(df_temp)}")
print(f"Intervalo de tempo: 0 a 24 horas")
print(f"Temperatura min: {T_medido.min():.2f}°C")
print(f"Temperatura máx: {T_medido.max():.2f}°C")
print(f"Temperatura média: {T_medido.mean():.2f}°C")

# Aplicar filtros de suavização
# Savitzky-Golay
T_savgol = savgol_filter(T_medido, window_length=15, polyorder=3)

# Média móvel
window = 5
T_media_movel = np.convolve(T_medido, np.ones(window)/window, mode='same')

# Remover drift
z = np.polyfit(t_array, T_medido, 1)
p = np.poly1d(z)
T_sem_drift = T_medido - p(t_array) + p(0)  # Remove drift mas mantém valor inicial

# Visualização
fig = plt.figure(figsize=(15, 10))
gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

# Painel 1: Dados brutos
ax = fig.add_subplot(gs[0, 0])
ax.plot(t_array, T_medido, 'b-', linewidth=1.5, alpha=0.7, label='Dados medidos')
ax.plot(t_array, df_temp['Temp_Ideal'], 'r--', linewidth=2, label='Sem ruído/drift')
ax.scatter(t_array[spike_indices], T_medido[spike_indices], s=100, c='red', marker='X',
          edgecolors='darkred', linewidth=2, label='Spikes', zorder=5)
ax.set_xlabel('Tempo (h)', fontsize=10)
ax.set_ylabel('Temperatura (°C)', fontsize=10)
ax.set_title('Dados Brutos: Ruído, Drift e Spikes', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Painel 2: Comparação de filtros
ax = fig.add_subplot(gs[0, 1])
ax.plot(t_array, T_medido, 'k-', linewidth=1, alpha=0.4, label='Original')
ax.plot(t_array, T_savgol, 'g-', linewidth=2.5, label='Savitzky-Golay')
ax.plot(t_array, T_media_movel, 'b--', linewidth=2, alpha=0.7, label='Média móvel')
ax.plot(t_array, df_temp['Temp_Ideal'], 'r:', linewidth=2, label='Ideal')
ax.set_xlabel('Tempo (h)', fontsize=10)
ax.set_ylabel('Temperatura (°C)', fontsize=10)
ax.set_title('Suavização: Comparação de Métodos', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Painel 3: Remoção de drift
ax = fig.add_subplot(gs[1, 0])
ax.plot(t_array, T_medido, 'b-', linewidth=1.5, alpha=0.6, label='Com drift')
ax.plot(t_array, T_sem_drift, 'g-', linewidth=2, label='Sem drift')
ax.plot(t_array, p(t_array), 'r--', linewidth=2, alpha=0.7, label=f'Drift removido')
ax.set_xlabel('Tempo (h)', fontsize=10)
ax.set_ylabel('Temperatura (°C)', fontsize=10)
ax.set_title('Remoção de Drift Linear', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Painel 4: Análise de frequência (espectro)
ax = fig.add_subplot(gs[1, 1])
fft = np.fft.fft(T_medido - T_medido.mean())
freq = np.fft.fftfreq(len(t_array), t_array[1] - t_array[0])
potencia = np.abs(fft)**2

idx_positivos = freq > 0
ax.semilogy(freq[idx_positivos], potencia[idx_positivos], 'b-', linewidth=2)
ax.set_xlabel('Frequência (1/h)', fontsize=10)
ax.set_ylabel('Potência (escala log)', fontsize=10)
ax.set_title('Espectro de Frequência', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(0, 0.2)

plt.suptitle('Análise de Série Temporal: Temperatura', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== EFEITOS DA SUAVIZAÇÃO ===")
print(f"Erro RMS (Savitzky-Golay): {np.sqrt(np.mean((T_savgol - df_temp['Temp_Ideal'])**2)):.4f}°C")
print(f"Erro RMS (Média móvel): {np.sqrt(np.mean((T_media_movel - df_temp['Temp_Ideal'])**2)):.4f}°C")
print(f"Erro RMS (Sem filtro): {np.sqrt(np.mean((T_medido - df_temp['Temp_Ideal'])**2)):.4f}°C")

## Exercício Proposto: Análise Completa de Dados Experimentais

**Desafio:**

A partir dos dados do pêndulo (Bloco A):
1. Remova os outliers
2. Calcule o comprimento médio de cada medida
3. Faça regressão linear de T² vs L para estimar g
4. Calcule o erro e a incerteza na estimativa de g

In [ ]:
# Seus passos aqui
# Dica: use df[~df['Outlier']], polyfit, e propagação de incerteza